# CrewAI Outreach Personalization Crew

**Project:** Multi-Agent B2B Sales Outreach Pipeline  
**Framework:** CrewAI (multi-agent orchestration)  
**LLM Backend:** Ollama (local, qwen3 model)  

---

## Overview

This notebook builds a **4-agent sales outreach crew** that automates the entire B2B prospecting workflow — from identifying ideal customer profiles to delivering hyper-personalized outreach messages.

### The Agent Roster

| # | Agent | Role | Responsibility |
|---|-------|------|---------------|
| 1 | **ICP Finder** | Ideal Customer Profile Analyst | Defines who to target based on product fit, firmographics, and buying signals |
| 2 | **Company Researcher** | Account Intelligence Analyst | Deep-dives into specific companies to uncover pain points, initiatives, and hooks |
| 3 | **Outreach Drafter** | Sales Copywriter | Writes cold outreach emails using research insights |
| 4 | **Personalization Agent** | Message Optimizer | Refines each email with company-specific details, timing hooks, and A/B variants |

### Task Handoff Architecture

```
Product/Service Description
         │
         ▼
  ┌──────────────┐
  │  ICP Finder  │  "WHO should we target?"
  └──────┬───────┘
         │ ICP criteria + target company list
         ▼
  ┌──────────────────┐
  │ Company Researcher│  "WHAT do we know about each target?"
  └──────┬───────────┘
         │ Company dossiers with pain points & hooks
         ▼
  ┌──────────────────┐
  │ Outreach Drafter │  "HOW do we reach them?"
  └──────┬───────────┘
         │ Draft outreach emails per company
         ▼
  ┌──────────────────────┐
  │ Personalization Agent│  "Make it IMPOSSIBLE to ignore"
  └──────────────────────┘
         │
         ▼
  Personalized emails + A/B variants + send-time recommendations
```

### How Task Handoff Works in CrewAI

**Task handoff** is the mechanism by which one agent's output becomes the next agent's input:

1. **Automatic context injection** — In sequential mode, CrewAI appends the previous task's output to the next task's prompt
2. **No explicit passing needed** — Agents don't call each other; the Crew orchestrator handles data flow
3. **Cumulative context** — Each agent sees ALL previous outputs, not just the immediately preceding one
4. **Output shaping** — The `expected_output` field on each task ensures the handoff data is in a usable format for the next agent

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q crewai crewai-tools langchain-ollama

In [ ]:
import os
import json
import textwrap
from datetime import datetime

from crewai import Agent, Task, Crew, Process
from crewai import LLM

print("CrewAI imports successful")
print(f"Timestamp: {datetime.now().isoformat()}")

## 2. LLM Configuration

All agents share the same local Ollama LLM. Each agent's unique behavior comes from its role, goal, and backstory — not from a different model.

In [ ]:
llm = LLM(
    model="ollama/qwen3",
    base_url="http://localhost:11434",
    temperature=0.7,
)

print(f"LLM configured: {llm.model}")

## 3. Define the Agents

### Agent Design Philosophy

In a sales outreach crew, each agent mirrors a role on a real sales team:

| Real-World Role | CrewAI Agent | Why Separate? |
|----------------|-------------|--------------|
| Sales Ops / RevOps | ICP Finder | Targeting strategy requires analytical, not creative, thinking |
| SDR Research | Company Researcher | Deep research is time-intensive and needs a focused specialist |
| SDR / Copywriter | Outreach Drafter | Writing outreach requires different skills than research |
| Growth Hacker | Personalization Agent | Personalization needs both research context AND copy to work with |

**Key principle:** Narrow agents outperform generalists. An agent that tries to research AND write produces mediocre results at both.

### 3.1 ICP Finder Agent

**Handoff role:** First in the pipeline — defines the targeting criteria that constrain all downstream work. If the ICP is wrong, every subsequent agent wastes effort on the wrong companies.

**What makes a good ICP backstory?** The agent needs to think in terms of firmographics (company size, industry, revenue), technographics (tech stack, tools), and buying signals (hiring patterns, funding rounds, pain points).

In [ ]:
icp_finder = Agent(
    role="Ideal Customer Profile Strategist",
    goal=(
        "Analyze the product/service offering and define a precise Ideal Customer "
        "Profile (ICP) including firmographic criteria, technographic signals, "
        "buying triggers, and a prioritized list of 5 target companies that match."
    ),
    backstory=(
        "You are a B2B go-to-market strategist with 10 years of experience building "
        "ICP frameworks for SaaS companies from seed stage to Series D. You think in "
        "terms of firmographics (industry, employee count, revenue range, geography), "
        "technographics (current tech stack, tools they already use), behavioral signals "
        "(recent funding, hiring surges, product launches), and pain indicators (public "
        "complaints, job postings mentioning specific challenges). You always produce "
        "ICPs with BOTH inclusion criteria (must-haves) and exclusion criteria (disqualifiers). "
        "Your target lists include specific company names with reasoning for each."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {icp_finder.role}")

### 3.2 Company Researcher Agent

**Handoff role:** Receives the ICP and target company list from the ICP Finder. Produces detailed dossiers that the Outreach Drafter will mine for personalization hooks.

**Critical handoff:** The quality of company research directly determines how personalized the outreach can be. Generic research → generic emails → low response rates.

In [ ]:
company_researcher = Agent(
    role="Account Intelligence Analyst",
    goal=(
        "Research each target company in depth to uncover recent news, strategic "
        "initiatives, technology decisions, pain points, key decision-makers, "
        "and specific angles that an outreach email could reference."
    ),
    backstory=(
        "You are a sales intelligence analyst who has built account dossiers for "
        "enterprise sales teams at companies like Salesforce, Gong, and Outreach.io. "
        "You know that effective research goes beyond Wikipedia summaries — you dig into "
        "recent earnings calls, press releases, job postings (which reveal strategic "
        "priorities), LinkedIn activity of key executives, technology adoption signals, "
        "and competitive pressures. For each company, you produce a structured dossier "
        "with: company overview, recent developments (last 6 months), identified pain "
        "points, key decision-makers with titles, and 2-3 specific 'hooks' — facts that "
        "a salesperson could reference to show they did their homework."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {company_researcher.role}")

### 3.3 Outreach Drafter Agent

**Handoff role:** Receives company dossiers and transforms research insights into compelling cold outreach emails. The drafter writes the initial structure; the personalizer refines.

**Why separate drafting from personalization?** Drafting is about email structure, value proposition framing, and CTA design. Personalization is about weaving in company-specific details. Combining them in one agent dilutes both skills.

In [ ]:
outreach_drafter = Agent(
    role="B2B Outreach Copywriter",
    goal=(
        "Write compelling, concise cold outreach emails for each target company. "
        "Each email should be under 150 words, reference specific company details, "
        "articulate a clear value proposition, and end with a low-friction CTA."
    ),
    backstory=(
        "You are a cold email specialist who has written outreach sequences that "
        "consistently achieve 15-25% open rates and 3-8% reply rates. You follow "
        "proven frameworks: Problem-Agitate-Solve (PAS), Before-After-Bridge (BAB), "
        "and AIDA (Attention-Interest-Desire-Action). Your emails are always under "
        "150 words because you know brevity wins in cold outreach. You never use "
        "generic openers like 'I hope this finds you well' — instead you lead with "
        "a specific observation about the recipient's company. Your CTAs are always "
        "low-friction ('Would a 15-min call next week make sense?' not 'Schedule a demo'). "
        "You write in a peer-to-peer tone, not a vendor-to-buyer tone."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {outreach_drafter.role}")

### 3.4 Personalization Agent

**Handoff role:** The **final agent** — receives the drafted emails and the original company research, then enhances each email with hyper-specific details, timing hooks, and A/B test variants.

**Why this agent exists:** Most cold emails fail because they're "personalized" with just a company name swap. Real personalization references specific initiatives, recent events, or pain points that only someone who researched the company would know.

In [ ]:
personalizer = Agent(
    role="Outreach Personalization Specialist",
    goal=(
        "Take each drafted email and enhance it with hyper-specific personalization: "
        "reference recent company events, align the value prop to known pain points, "
        "add timing hooks, suggest optimal send times, and create an A/B variant "
        "with a different angle for each email."
    ),
    backstory=(
        "You are a personalization expert who has optimized outreach campaigns for "
        "100+ B2B companies, improving reply rates by 2-4x through deep personalization. "
        "You know that personalization is NOT just inserting {{company_name}} — it's about "
        "demonstrating genuine understanding. You add three layers of personalization: "
        "(1) Company-level — reference a recent initiative, funding round, or product launch; "
        "(2) Role-level — align the message to the recipient's likely priorities based on title; "
        "(3) Timing-level — connect to something happening NOW (earnings season, fiscal year end, "
        "industry event). For each email, you produce: the enhanced version, an A/B variant "
        "with a completely different angle, and a 'why this works' annotation explaining "
        "the personalization choices."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {personalizer.role}")

### Agent Roster Summary

In [ ]:
agents = [icp_finder, company_researcher, outreach_drafter, personalizer]

print("=" * 70)
print("OUTREACH CREW ROSTER")
print("=" * 70)
for i, agent in enumerate(agents, 1):
    print(f"\n[{i}] {agent.role}")
    print(f"    Goal: {agent.goal[:90]}...")
    print(f"    Delegation: {'Enabled' if agent.allow_delegation else 'Disabled'}")
print("\n" + "=" * 70)
print(f"Total agents: {len(agents)}")

## 4. Define Tasks & Handoff Chain

### Understanding Task Handoff

In CrewAI, **task handoff** is the mechanism that connects agents:

```
┌─────────┐   ICP doc    ┌────────────┐  dossiers   ┌───────────┐  drafts    ┌──────────────┐
│ ICP     ├─────────────>│ Company    ├────────────>│ Outreach  ├──────────>│ Personalizer │
│ Finder  │  (handoff 1) │ Researcher │ (handoff 2) │ Drafter   │(handoff 3)│              │
└─────────┘              └────────────┘             └───────────┘           └──────────────┘
```

**What happens at each handoff:**

| Handoff | From → To | What's Passed | Why It Matters |
|---------|-----------|--------------|----------------|
| 1 | ICP Finder → Researcher | ICP criteria + 5 target companies | Constrains research scope |
| 2 | Researcher → Drafter | Company dossiers with pain points & hooks | Gives the writer material to work with |
| 3 | Drafter → Personalizer | Draft emails + (cumulative research context) | Raw material for personalization |

**Cumulative context:** By handoff 3, the personalizer can see the ICP criteria (from Task 1), the company dossiers (from Task 2), AND the draft emails (from Task 3). This accumulated context enables deep personalization.

### 4.1 Configure the Product/Service

The pipeline is parameterized — change the product definition and re-run for any B2B offering.

In [ ]:
# =====================================================
# CONFIGURE YOUR PRODUCT/SERVICE HERE
# =====================================================
PRODUCT = "DataSync Pro — an AI-powered data integration platform that automates ETL pipelines, reduces data engineering workload by 70%, and ensures real-time data freshness across warehouses"
VALUE_PROP = "Eliminates manual ETL scripting, cuts data pipeline maintenance costs, and delivers real-time analytics-ready data"
PRICE_RANGE = "$2,000-$15,000/month depending on data volume"
SALES_MOTION = "Product-led growth with enterprise upsell; 14-day free trial available"
# =====================================================

print(f"Product: {PRODUCT[:80]}...")
print(f"Value Prop: {VALUE_PROP[:80]}...")
print(f"Price Range: {PRICE_RANGE}")
print(f"Sales Motion: {SALES_MOTION}")

### 4.2 Task 1 — ICP Definition

**Agent:** ICP Finder  
**Input:** Product description + value proposition  
**Output:** ICP document + 5 ranked target companies  
**Handoff to:** Company Researcher (who uses the target list to focus research)

In [ ]:
task_icp = Task(
    description=(
        f"Define the Ideal Customer Profile for this product:\n\n"
        f"PRODUCT: {PRODUCT}\n"
        f"VALUE PROPOSITION: {VALUE_PROP}\n"
        f"PRICE RANGE: {PRICE_RANGE}\n"
        f"SALES MOTION: {SALES_MOTION}\n\n"
        "Your ICP document MUST include:\n"
        "1. **Firmographic Criteria**: Industry verticals, company size (employees + revenue), geography\n"
        "2. **Technographic Signals**: Tools/platforms they likely use, tech maturity indicators\n"
        "3. **Buying Triggers**: Events that create urgency (funding, hiring, tech migration)\n"
        "4. **Pain Indicators**: Problems they're likely experiencing that our product solves\n"
        "5. **Decision-Maker Personas**: Titles/roles involved in the buying decision\n"
        "6. **Exclusion Criteria**: Who should we NOT target (and why)\n"
        "7. **Target Company List**: 5 specific companies that match, ranked by fit score (1-10), "
        "   with a 2-sentence justification for each\n"
    ),
    expected_output=(
        "A structured ICP document with firmographic/technographic criteria, buying triggers, "
        "pain indicators, decision-maker personas, exclusions, and a ranked list of 5 "
        "target companies with fit scores and justifications."
    ),
    agent=icp_finder,
)

print(f"Task created: ICP Definition -> {task_icp.agent.role}")

### 4.3 Task 2 — Company Research

**Agent:** Company Researcher  
**Input:** ICP document + target company list (from Task 1 handoff)  
**Output:** Detailed dossier for each company  
**Handoff to:** Outreach Drafter (who mines dossiers for email hooks)

**Handoff quality check:** The dossier format is designed to make the drafter's job easy — each company gets a structured profile with pre-identified hooks that can be directly referenced in emails.

In [ ]:
task_research = Task(
    description=(
        "Using the ICP and target company list from the previous agent, "
        "research each of the 5 target companies in depth.\n\n"
        "For EACH company, produce a dossier covering:\n"
        "1. **Company Overview**: What they do, size, revenue, key products\n"
        "2. **Recent Developments**: Last 6 months — funding, launches, partnerships, leadership changes\n"
        "3. **Technology Stack**: Known tools, platforms, and infrastructure choices\n"
        "4. **Pain Points**: Specific challenges related to our product's value proposition\n"
        "5. **Key Decision-Makers**: 2-3 people with titles who would evaluate our product\n"
        "6. **Outreach Hooks**: 2-3 specific facts/events that an SDR could reference in an email "
        "   to demonstrate genuine knowledge of the company\n"
        "7. **Competitive Context**: What alternatives they might be evaluating\n"
        "8. **Fit Score**: Rate 1-10 how well this company matches the ICP (with reasoning)\n"
    ),
    expected_output=(
        "5 structured company dossiers, each with overview, recent developments, "
        "tech stack, pain points, decision-makers, outreach hooks, competitive "
        "context, and fit score. Format as clearly separated profiles."
    ),
    agent=company_researcher,
)

print(f"Task created: Company Research -> {task_research.agent.role}")

### 4.4 Task 3 — Outreach Drafting

**Agent:** Outreach Drafter  
**Input:** Company dossiers (from Task 2 handoff) + ICP context (from Task 1)  
**Output:** Draft cold emails for each company  
**Handoff to:** Personalization Agent (who refines with deeper personalization)

**Handoff design:** The drafter produces structured emails with clearly marked sections (subject, opening, body, CTA) so the personalizer can surgically improve each part.

In [ ]:
task_outreach = Task(
    description=(
        f"Write cold outreach emails for each of the 5 researched companies.\n\n"
        f"Product: {PRODUCT[:60]}...\n"
        f"Value Prop: {VALUE_PROP}\n\n"
        "For EACH company, write an email that:\n"
        "1. **Subject Line**: Under 50 characters, creates curiosity (no clickbait)\n"
        "2. **Opening Line**: Reference a specific fact about the company (from research)\n"
        "3. **Problem Statement**: Articulate a pain point they likely have\n"
        "4. **Value Bridge**: Connect the pain to our product's solution (1-2 sentences)\n"
        "5. **Social Proof**: One brief credibility signal (metric, customer, or achievement)\n"
        "6. **CTA**: Low-friction ask (15-min call, not a demo)\n"
        "7. **Total Length**: Under 150 words\n\n"
        "Use the PAS (Problem-Agitate-Solve) framework for 3 emails and "
        "BAB (Before-After-Bridge) for 2 emails. Label which framework each uses."
    ),
    expected_output=(
        "5 cold outreach emails, one per company, each with subject line, body (under "
        "150 words), and labeled framework (PAS or BAB). Clearly separated by company name."
    ),
    agent=outreach_drafter,
)

print(f"Task created: Outreach Drafting -> {task_outreach.agent.role}")

### 4.5 Task 4 — Personalization & Optimization

**Agent:** Personalization Agent  
**Input:** Draft emails (Task 3) + company dossiers (Task 2) + ICP (Task 1) — cumulative context  
**Output:** Enhanced emails + A/B variants + send-time recommendations  

**This is where cumulative handoff shines.** The personalizer has access to:
- The ICP framework (from Task 1) → understands what matters strategically
- The company dossiers (from Task 2) → knows each company's specific situation
- The draft emails (from Task 3) → has the base copy to enhance

No single agent could produce this output alone — it requires the cumulative intelligence of the entire pipeline.

In [ ]:
task_personalize = Task(
    description=(
        "Enhance each of the 5 outreach emails with deep personalization.\n\n"
        "For EACH email, produce:\n"
        "1. **Enhanced Version (A)**: The original email improved with:\n"
        "   - A more specific opening line referencing a RECENT company event\n"
        "   - Pain point language aligned to the recipient's likely role/title\n"
        "   - A timing hook (why reaching out NOW makes sense)\n"
        "   - Adjusted tone for the company's culture (formal for enterprise, casual for startups)\n\n"
        "2. **A/B Variant (B)**: A completely different angle for the same company:\n"
        "   - Different subject line\n"
        "   - Different opening hook (e.g., if A leads with a problem, B leads with a result)\n"
        "   - Different CTA\n\n"
        "3. **Personalization Annotations**: For each email, explain:\n"
        "   - What personalization layers were added (company/role/timing)\n"
        "   - Why each choice was made\n"
        "   - Recommended send day/time based on the recipient's likely schedule\n"
        "   - Suggested follow-up timing (days after first email)\n"
    ),
    expected_output=(
        "For each of the 5 companies: (A) enhanced email, (B) A/B variant email, "
        "and (C) personalization annotations with reasoning, send-time recommendation, "
        "and follow-up timing. All emails remain under 150 words."
    ),
    agent=personalizer,
)

print(f"Task created: Personalization -> {task_personalize.agent.role}")

### Task Handoff Visualization

In [ ]:
tasks = [task_icp, task_research, task_outreach, task_personalize]
task_names = ["ICP Definition", "Company Research", "Outreach Drafting", "Personalization"]
handoff_labels = [
    "ICP criteria + 5 target companies",
    "5 company dossiers with hooks",
    "5 draft emails (PAS/BAB)",
]

print("TASK HANDOFF CHAIN")
print("=" * 70)
for i, (name, task) in enumerate(zip(task_names, tasks), 1):
    print(f"  Task {i}: {name}")
    print(f"    Agent: {task.agent.role}")
    if i <= len(handoff_labels):
        print(f"    Handoff {i}: passes '{handoff_labels[i-1]}'")
        print(f"    {'':12}|")
    else:
        print(f"    >> FINAL OUTPUT: personalized emails + A/B variants")
print("=" * 70)
print(f"\nContext accumulation: Task 4 sees outputs from Tasks 1 + 2 + 3")

## 5. Assemble and Run the Crew

### Crew Assembly & Handoff Mechanics

When you create a `Crew` with `Process.sequential`:

1. The crew takes the task list in order: `[task_icp, task_research, task_outreach, task_personalize]`
2. It runs Task 1 (ICP Finder), collects the output
3. It appends Task 1's output as context to Task 2's prompt, then runs Task 2
4. It appends Tasks 1+2 outputs as context to Task 3, then runs Task 3
5. It appends Tasks 1+2+3 outputs as context to Task 4, then runs Task 4

**Each agent operates in its own "sandbox"** — it doesn't know about other agents, only about the accumulated context it receives. This isolation makes agents composable and reusable.

In [ ]:
crew = Crew(
    agents=agents,
    tasks=tasks,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Crew assembled: {len(crew.agents)} agents, {len(crew.tasks)} tasks")
print(f"Process: {crew.process}")
print(f"Memory: {'Enabled' if crew.memory else 'Disabled'}")

In [ ]:
# Execute the outreach pipeline
print("=" * 70)
print(f"LAUNCHING OUTREACH CREW — {datetime.now().strftime('%H:%M:%S')}")
print(f"Product: {PRODUCT[:60]}...")
print(f"Pipeline: ICP -> Research -> Draft -> Personalize")
print("=" * 70)

result = crew.kickoff()

print("\n" + "=" * 70)
print(f"CREW COMPLETED — {datetime.now().strftime('%H:%M:%S')}")
print("=" * 70)

## 6. Analyze Results

### 6.1 Final Output — Personalized Outreach Package

The crew's final result is the personalizer's output: enhanced emails with A/B variants and annotations.

In [ ]:
print("=" * 70)
print("FINAL OUTPUT — Personalized Outreach Package")
print("=" * 70)
print(result.raw)

### 6.2 Trace the Handoff Chain

Inspect each agent's individual output to see how information flowed through the pipeline.

In [ ]:
for name, task in zip(task_names, tasks):
    print("\n" + "=" * 70)
    print(f"HANDOFF OUTPUT: {name}")
    print(f"Agent: {task.agent.role}")
    print("=" * 70)
    if task.output:
        text = task.output.raw
        if len(text) > 2000:
            print(text[:2000])
            print(f"\n... [{len(text) - 2000} more characters]")
        else:
            print(text)
    else:
        print("[No output captured]")

### 6.3 Pipeline Metrics

In [ ]:
print("OUTREACH PIPELINE METRICS")
print("=" * 60)
print(f"{'Stage':<22} {'Agent':<32} {'Output':>8}")
print("-" * 60)

total = 0
for name, task in zip(task_names, tasks):
    length = len(task.output.raw) if task.output else 0
    total += length
    print(f"{name:<22} {task.agent.role:<32} {length:>6,}")

print("-" * 60)
print(f"{'TOTAL':<54} {total:>6,}")

if hasattr(result, 'token_usage') and result.token_usage:
    usage = result.token_usage
    print(f"\nTokens: {usage.get('total_tokens', 'N/A')}")
else:
    print("\nToken usage: not tracked (typical with local Ollama)")

## 7. Save Outputs

In [ ]:
# Save the complete pipeline output
report_lines = [
    f"# Outreach Pipeline Report: {PRODUCT[:60]}",
    f"**Generated:** {datetime.now().isoformat()}",
    f"**Value Prop:** {VALUE_PROP}",
    "---",
]

for name, task in zip(task_names, tasks):
    report_lines.append(f"\n## {name} (by {task.agent.role})\n")
    report_lines.append(task.output.raw if task.output else "[No output]")
    report_lines.append("\n---")

report = "\n".join(report_lines)

with open("outreach_pipeline_report.md", "w", encoding="utf-8") as f:
    f.write(report)

print(f"Report saved: outreach_pipeline_report.md ({len(report):,} chars)")

## 8. Experiment: Different Product

Demonstrate pipeline reusability by targeting a completely different B2B product.

In [ ]:
# =====================================================
# SECOND PRODUCT
# =====================================================
PRODUCT_2 = "SecureDoc AI — an AI-powered document compliance platform that automatically classifies sensitive documents, enforces retention policies, and generates audit-ready compliance reports"
VALUE_PROP_2 = "Automates document classification and compliance, reducing legal risk and audit prep time by 80%"
PRICE_RANGE_2 = "$5,000-$25,000/month based on document volume and regulatory frameworks"
SALES_MOTION_2 = "Enterprise direct sales; requires security review and legal sign-off"

print(f"Product 2: {PRODUCT_2[:80]}...")
print(f"Value Prop: {VALUE_PROP_2[:80]}...")

In [ ]:
# Build task set for the second product
task2_icp = Task(
    description=(
        f"Define the ICP for: {PRODUCT_2}\n"
        f"Value Prop: {VALUE_PROP_2}\n"
        f"Price: {PRICE_RANGE_2}\n"
        f"Sales Motion: {SALES_MOTION_2}\n\n"
        "Include firmographics, technographics, buying triggers, pain indicators, "
        "exclusion criteria, and 5 ranked target companies with fit scores."
    ),
    expected_output="ICP document with criteria and 5 target companies.",
    agent=icp_finder,
)

task2_research = Task(
    description=(
        "Research each of the 5 target companies from the ICP. For each: "
        "company overview, recent developments, tech stack, pain points, "
        "key decision-makers, 2-3 outreach hooks, and fit score."
    ),
    expected_output="5 company dossiers with hooks and decision-maker profiles.",
    agent=company_researcher,
)

task2_outreach = Task(
    description=(
        f"Write cold emails for each company. Product: {PRODUCT_2[:50]}... "
        f"Value: {VALUE_PROP_2[:50]}... "
        "Under 150 words each. Use PAS for 3 emails, BAB for 2. "
        "Include subject, opening hook, problem, value bridge, proof, CTA."
    ),
    expected_output="5 cold emails with subject lines, labeled frameworks.",
    agent=outreach_drafter,
)

task2_personalize = Task(
    description=(
        "Personalize each email: enhance with recent company events, "
        "role-level language, timing hooks. Create A/B variant for each. "
        "Add annotations explaining personalization choices and send-time recs."
    ),
    expected_output="5 enhanced emails + 5 A/B variants + annotations.",
    agent=personalizer,
)

tasks_2 = [task2_icp, task2_research, task2_outreach, task2_personalize]
print(f"Second product tasks created: {len(tasks_2)} tasks")

In [ ]:
crew2 = Crew(
    agents=agents,
    tasks=tasks_2,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Launching Crew 2 — {datetime.now().strftime('%H:%M:%S')}")
print(f"Product: {PRODUCT_2[:50]}...")
print("=" * 70)

result2 = crew2.kickoff()

print(f"\nCrew 2 completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
print("=" * 70)
print(f"PERSONALIZED OUTREACH — {PRODUCT_2[:50]}...")
print("=" * 70)
print(result2.raw)

## 9. Compare Both Runs

In [ ]:
print("PIPELINE COMPARISON")
print("=" * 70)
print(f"{'Stage':<22} {'DataSync (chars)':<20} {'SecureDoc (chars)':<20}")
print("-" * 70)

for name, t1, t2 in zip(task_names, tasks, tasks_2):
    len1 = len(t1.output.raw) if t1.output else 0
    len2 = len(t2.output.raw) if t2.output else 0
    print(f"{name:<22} {len1:<20,} {len2:<20,}")

total1 = sum(len(t.output.raw) for t in tasks if t.output)
total2 = sum(len(t.output.raw) for t in tasks_2 if t.output)
print("-" * 70)
print(f"{'TOTAL':<22} {total1:<20,} {total2:<20,}")

## 10. Deep Dive: Handoff Patterns & Best Practices

### Handoff Pattern 1: Linear Chain (This Notebook)

```
ICP -> Research -> Draft -> Personalize
```

Each agent sees ALL previous outputs. Simple and predictable.

**When to use:** When tasks have a clear sequential dependency — research must happen before writing, writing before personalization.

### Handoff Pattern 2: Explicit Context Control

```python
# Instead of automatic context, specify exactly what each task sees
task_personalize = Task(
    ...,
    context=[task_research, task_outreach],  # Sees research + drafts, skips ICP
)
```

The `context` parameter overrides automatic handoff. Use this when an agent doesn't need ALL previous context — only specific tasks.

### Handoff Pattern 3: Parallel Fan-Out

```
                    ┌-> Outreach Drafter (email)
Research -> ICP -> ─┤
                    └-> Social Drafter  (LinkedIn InMail)
```

Multiple agents receive the same handoff and work in parallel. Both drafters get the same research but produce different formats.

### Handoff Pattern 4: Convergent Fan-In

```
Email Drafter  ──┐
                  ├──> Personalizer (receives both)
Social Drafter ──┘
```

The personalizer receives outputs from multiple upstream agents and synthesizes them.

### Handoff Quality Principles

| Principle | Why It Matters | How to Implement |
|-----------|---------------|-----------------|
| **Structured output** | Next agent can parse reliably | Use `expected_output` with specific format requirements |
| **Labeled sections** | Downstream agents find relevant info | Ask for numbered sections, headers, or JSON |
| **Completeness** | Missing data can't be recovered later | Task descriptions should enumerate ALL required fields |
| **Conciseness** | Too much context drowns the signal | Ask for summaries + details, not stream-of-consciousness |

## 11. Advanced: Explicit Context Handoff Demo

Demonstrate how `context` parameter gives fine-grained control over which task outputs each agent receives.

In [ ]:
# Explicit context demo: personalizer ONLY sees research + drafts (skips ICP)

task_ctx_icp = Task(
    description=(
        f"Define ICP for {PRODUCT[:40]}... "
        "Include firmographics, buying triggers, and 5 target companies."
    ),
    expected_output="ICP document with 5 target companies.",
    agent=icp_finder,
)

task_ctx_research = Task(
    description="Research the 5 target companies. Provide dossiers with hooks.",
    expected_output="5 company dossiers.",
    agent=company_researcher,
    context=[task_ctx_icp],  # Explicit: sees ICP output
)

task_ctx_draft = Task(
    description=(
        f"Write cold emails for each company. Product: {PRODUCT[:40]}... "
        "Under 150 words each."
    ),
    expected_output="5 cold outreach emails.",
    agent=outreach_drafter,
    context=[task_ctx_research],  # Explicit: sees ONLY research (not ICP directly)
)

task_ctx_personalize = Task(
    description=(
        "Personalize each email. Create A/B variants with annotations."
    ),
    expected_output="Enhanced emails + A/B variants.",
    agent=personalizer,
    context=[task_ctx_research, task_ctx_draft],  # Explicit: sees research + drafts
)

ctx_tasks = [task_ctx_icp, task_ctx_research, task_ctx_draft, task_ctx_personalize]

print("EXPLICIT CONTEXT HANDOFF MAP")
print("=" * 60)
for i, task in enumerate(ctx_tasks, 1):
    ctx_sources = [t.agent.role for t in (task.context or [])]
    ctx_str = ", ".join(ctx_sources) if ctx_sources else "auto (previous task)"
    print(f"  Task {i}: {task.agent.role}")
    print(f"    Receives context from: {ctx_str}")
    if i < len(ctx_tasks):
        print(f"    {'':4}|")

print("\nNote: Task 4 skips the ICP output — it only needs research + drafts.")

In [ ]:
# Run the explicit-context crew
crew_ctx = Crew(
    agents=agents,
    tasks=ctx_tasks,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Launching context-controlled crew — {datetime.now().strftime('%H:%M:%S')}")
result_ctx = crew_ctx.kickoff()
print(f"\nContext-controlled crew completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
# Show what each agent received vs produced
print("CONTEXT-CONTROLLED HANDOFF RESULTS")
print("=" * 60)
ctx_names = ["ICP Definition", "Company Research", "Outreach Drafting", "Personalization"]
for name, task in zip(ctx_names, ctx_tasks):
    length = len(task.output.raw) if task.output else 0
    ctx_sources = [t.agent.role for t in (task.context or [])]
    ctx_str = ", ".join(ctx_sources) if ctx_sources else "none (first task)"
    print(f"  {name:<22} {length:>6,} chars | Context: {ctx_str}")

## 12. Key Takeaways

### What We Built
- A **4-agent outreach pipeline** (ICP Finder → Company Researcher → Outreach Drafter → Personalization Agent)
- Ran it on **two different B2B products** (DataSync Pro + SecureDoc AI) to demonstrate reusability
- Demonstrated **explicit context handoff** with the `context` parameter

### Task Handoff Concepts
1. **Automatic handoff** — In sequential mode, each task's output is appended to the next task's context
2. **Cumulative context** — Later agents see ALL previous outputs, enabling deep personalization
3. **Explicit context** — The `context` parameter gives precise control over which outputs each agent sees
4. **Output shaping** — `expected_output` ensures handoff data is structured for downstream consumption

### Agent Design Lessons
- **ICP Finder** produces structured criteria → constrains the researcher's scope
- **Company Researcher** returns dossiers with pre-identified hooks → drafter has ready-made material
- **Outreach Drafter** uses proven frameworks (PAS, BAB) → consistent email quality
- **Personalizer** layers three types of personalization (company/role/timing) → maximum relevance

### Handoff Quality Checklist
- [ ] Each task's `expected_output` defines the format the next agent needs
- [ ] Research tasks include "hooks" that writers can directly reference
- [ ] Draft tasks produce clearly sectioned output (subject, opening, body, CTA)
- [ ] The final agent has access to enough upstream context for deep personalization

### Production Tips
- Add web search tools to the Company Researcher for real-time data
- Use `output_pydantic` to enforce structured output schemas at each handoff
- Enable `memory=True` when running the crew daily to learn from past campaigns
- Add a **Compliance Checker** agent before the personalizer for regulated industries
- Set up callbacks to log each handoff's output for debugging pipeline quality